# **COLA: library for CE with joint-distribution-informed shapley towards actionable minimality**

**Library paper:** [xai-cola: A python library for sparsifying counterfactual
explanations](https://www.scholar-inbox.com/papers/Zhu2026ARXIV_xai_cola_A_Python.pdf)

**Paper:** [Refining Counterfactual Explanations With Joint-Distribution-Informed Shapley Towards Actionable Minimality](https://arxiv.org/pdf/2410.05419)  

**Github:** [github.com/understanding-ml/COLA](https://github.com/understanding-ml/COLA)

<a id='theory'></a>

## 1. **Theory**

The title contains many scary words, but the scariest one, I’m sure, is `joint-distribution-informed shapley`. What are Shapley values informed by the joint distribution? Let’s go through the prerequisites. But let’s start from the beginning.

### **1.1 Why do we need counterfactual explanations?**

Imagine a system that predicts your weight in 3 years based on a set of features. You enter your data and, with a starting weight of 60 kg, you receive a prediction — 80 kg. That sounds unpleasant and depressing, and you’re not happy about it. Why should your weight go up, and what is the minimal thing you could change to stay fit and amazing?

This is where a **counterfactual explanation (CE)** would help. Based on the fact that you are 24, you recently quit smoking, and you have noticed a tendency lately to “snack”, the minimal change could look like this:
*"If you record exercise as 4 times per week instead of 1, the predicted weight will not exceed 65 kg."*

Now it’s clear what to do. Hooray!

### **1.2 How are things going?**

In the simplest case, the task of finding a counterfactual (that closest instance with a different prediction) can be formulated as a nearest neighbor search problem. This heuristic is both good — because it is intuitive — and bad — because the nearest neighbor will not necessarily correspond to the minimal set of changes and may contain some noise.

#### **1.2.1 What exists?**

Among the libraries that allow implementation of Counterfactual Explanations, one of the most popular is [DiCE](https://github.com/interpretml/DiCE/tree/main). In it, the query for a counterfactual explanation is formulated as:

* In the entire dataset (which we will provide to you), find the nearest counterfactuals for a chosen instance such that the prediction $\hat{f}(x)$ for it $\in [y_i, y_j]$

There are other algorithms as well, but in general the search for counterfactuals resembles the question above, with variations in the neighbor search method.

#### **1.2.1 What’s wrong?**

* Several counterfactual explanations may be found for the same case, and they may even contradict each other (the Rashomon effect).
* The discovered counterfactuals are not guaranteed to contain minimal changes.

The first problem cannot be solved. And no, I didn’t make a mistake when formulating that sentence :) But the second one can be addressed — and various algorithms are aimed exactly at this. However, until recently there were no convenient wrappers that allowed this to be used easily. And here it is — COLA.

### **1.3 COLA problem formulation and why it’s not straightforward**

Let’s look at why finding a counterfactual is not so simple. We have:

**Given:** factual data $\mathbf{x} \in \mathbb{R}^{n \times d}$, an ML model $f$, desired outcome $\mathbf{y}^*$
**Find:** a vector corresponding to prediction $\mathbf{y}^*$, with the minimal number of changed features and as close as possible to the original object according to metric $D$:

$$\min_{\mathbf{c}, \mathbf{z}} ; D(f(\mathbf{z}), \mathbf{y}^*)$$

If we remove the letter $c$, the above becomes a completely classical counterfactual search problem with requirement (1):

* (1) $D(\mathbf{z}, \mathbf{x}) \leq \varepsilon$ — the counterfactual is close to the original instance

However, if we introduce a constraint on the number of changes, the following aspects appear. We minimize such that:

* $\sum_{i,k} c_{ik} \leq C$ — we have a change “budget”. The budget is measured as the number of modifications, meaning $c_{ik} \in {0, 1}$ depending on whether a change is absent or present.
* $x_{ik}(1-c_{ik}) - Mc_{ik} \leq z_{ik} \leq x_{ik}(1-c_{ik}) + Mc_{ik}$, meaning that when a change is present ($c_{ik} =1$), then $z_{ik} \in [-M, M]$ for some constant $M$, allowing the feature to vary quite widely across possible values.

**Problem: in this formulation we face an NP-hard problem.**

Why? Let’s consider a simple case where $d=1$, and a linear model of the form:
$$y = \mathbf{W}\mathbf{x} + b$$

* $x \in \mathbb{R}^{n}$ — a vector of $n$ examples with one feature

* $\mathbf{W} \in \mathbb{R}^{m \times n}$ — a weight matrix

Let the vector $\mathbf{x} = 0$, and $D$ be the Euclidean metric. We write the problem of finding the best set of counterfactuals:

$$\min_{\mathbf{c}, \mathbf{z}} ; ||Wx - \mathbf{y}^*||_2^2,$$ with constraint (1) of the form:
$$||x-z||_2^2 = ||z||_2^2 \leq \varepsilon$$

Since the original $x = 0$, and we want to change the smallest possible number of features, the problem reduces to finding a vector $z : ||z||_0 < K$, where $K$ is the maximum number of non-zero elements.

Thus, the vector $z$ in our case has $n$ coordinates, and we choose which of these $n$ coordinates to make non-zero — this is the classical **Sparse Regression** or **Subset Selection** problem, and it is NP-hard because:

* We need to choose a subset of indices $S \subset {1,\dots,n}, \quad |S| \le K$
* For each such subset, solve a standard linear regression on the selected coordinates.
* Choose the best solution among all possible subsets.

The number of possible subsets: $\binom{n}{K}$, which grows exponentially with $n$. Already for $n=100$ and $K=10$:

$\binom{100}{10} \approx 1.7 \times 10^{13}$

Quite sad.


### **1.4 The idea of COLA: p-SHAP**

From the formulation above, it is clear that the key bottleneck here is the need to enumerate all features. What if we could somehow determine a smaller subset of features in advance? This intuition is exactly what COLA implements.

#### **1.4.1 Reminder: what is a Shapley value?**

For a function $f(x)$ and a set of features $N$, the contribution of feature $i$ is defined as:

$$\phi_i(f,x) = \sum_{S \subseteq N \setminus {i}} \frac{|S|!(|N|-|S|-1)!}{|N|!} \left[ f(x_{S \cup {i}}) - f(x_S) \right]$$

where:

* $S$ — a subset of features (also called a coalition, where feature $i$ is not present),
* $x_S$ — a vector where features outside $S$ are “turned off”, and $x_{S \cup {i}}$ — the same vector but now also including feature $i$,
* $f(\cdot)$ — the model prediction.

#### **1.4.2 Joint-distribution Shap**

Computing Shapley values is also unpleasant. Moreover, locally they may vary from example to example. First of all, to define a Shapley value over the whole dataset, the authors introduce conditional expectations:

$$v(S) = \mathbb{E}[f(X) \mid X_S = x_S]$$

That is, the value of the coalition function $S$ is the conditional expectation of the model given that the features in $S$ are fixed. In other words, it answers the question: if the features from $S$ are fixed as in the current object, while the remaining features follow the same distribution as in the real data, what is the model’s average prediction?

Then the Shapley value becomes:

$$\phi_i = \sum_{S \subseteq N \setminus {i}} \frac{|S|!(|N|-|S|-1)!}{|N|!} \Big( \mathbb{E}[f(X)\mid X_{S \cup {i}}] - \mathbb{E}[f(X)\mid X_S]\Big)$$

**Note that it is still computed relative to the object’s features, but the remaining features are sampled according to the same distribution as in the dataset.**

#### **1.4.3 Building COLA**

At this point we have gathered everything needed to understand where the optimization in COLA comes from. We:

1. Construct any approximate counterfactual.
2. Compute Shapley values for the current object.
3. Rank features by their contribution.
4. Modify only the top-$p$ features.
5. Obtain a sparse representation.

In other words, the counterfactual is constructed by allowing changes only in the $p$ most “influential” features. Therefore, we no longer need to enumerate subsets — the subset of the most influential features is fixed in advance.

Let’s go to the python-library!

<a id='install'></a>
## **2. Installation and import**

In [ ]:
# install (and then restart your runtime)
!pip install -q shap dice-ml
!pip install -q --no-deps xai-cola

In [ ]:
!pip install cem -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, HTML

# Sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# COLA
from xai_cola.datasets.german_credit import GermanCreditDataset
from xai_cola.ce_sparsifier.data import COLAData
from xai_cola.ce_sparsifier.models import Model
from xai_cola.ce_generator import DiCE
from xai_cola.ce_sparsifier import COLA

# Plots
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')


<a id='data'></a>

## 3. Data Loading and Model Training

To use the full functionality, we will work with the built-in **German Credit** dataset — a classic dataset for the credit scoring task. In this case, our **task** is to predict a client’s credit risk (0 = good, 1 = bad).

Therefore, the **goal of CE** is to find changes that move a client from class 1 to class 0 (credit approval).


In [ ]:
# Load dataset
dataset = GermanCreditDataset()
X_train, y_train, X_test, y_test = dataset.get_original_train_test_split()

print(f"Train: {X_train.shape}")
print(f"Test :  {X_test.shape}")
print(f"Class distribution (train):")
print(y_train.value_counts().to_string())

print("First 5 samples:")
display(X_train.head())

1. We have standard features: age, gender, job type, type of housing (rent/own), the status of savings and checking accounts, credit amount, loan duration, and loan purpose.
2. Risk refers to the probability that the client will not repay the loan. This is an imbalanced classification problem.
3. The class itself is implemented in a rather unusual way. For some reason, preprocessing happens in the initializer rather than internally in the pipeline, and at the same time it does not allow loading data with NaNs. Let’s leave that on the library’s conscience.


In [ ]:
# Features
numerical_features = ['Age', 'Credit amount', 'Duration']
categorical_features = ['Sex', 'Job', 'Housing', 'Saving accounts',
                        'Checking account', 'Purpose']


# Fast EDA
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, numerical_features):
    X_train.groupby(y_train)[feat].plot.hist(ax=ax, alpha=0.6, bins=20, legend=True)
    ax.set_title(f'{feat} per class')
    ax.set_xlabel(feat)
    ax.legend(['Risk=0 (хор.)', 'Risk=1 (плох.)'])
plt.tight_layout()
plt.suptitle('Numerical features distribution', y=1.02, fontsize=14)
plt.show()

Basically, there are no obvious skews in the features. We could run statistical tests or draw boxplots, but for the sake of speed we’ll skip that here.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# Categorical features preprocessing
for c in categorical_features:
    X_train[c] = X_train[c].astype(str)
    X_test[c] = X_test[c].astype(str)

# We will apply standardization (since the numerical features are on different scales) and OHE.
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
], remainder='passthrough')


# as the main model, we will use logistic regression (simple and interpretable)
lr_classifier = LogisticRegression(
    max_iter=1000, C=1.0,
    class_weight='balanced',
    random_state=42, solver='lbfgs'
)

# we will also train a decision tree — you can later rerun the pipeline with it
d_tree = RandomForestClassifier(random_state=42, max_depth=5)

# Pipeline: preprocessing + classifier
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', lr_classifier)
])

pipe_tree = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', d_tree)
])

pipe.fit(X_train, y_train)
pipe_tree.fit(X_train, y_train)

# Evaluation
y_pred = pipe.predict(X_test)
y_pred_tree = pipe_tree.predict(X_test)
print("Test report (lr model):")
print(classification_report(y_test, y_pred, target_names=['Risk=0 (хор.)', 'Risk=1 (плох.)']))

print("Test report (tree model):")
print(classification_report(y_test, y_pred_tree, target_names=['Risk=0 (хор.)', 'Risk=1 (плох.)']))

<a id='dice'></a>

## 4. Plain DiCE without COLA (baseline)

**DiCE** (Diverse Counterfactual Explanations) is a popular instance-wise CE algorithm.
It generates a set of counterfactuals that change the model’s prediction, but it may modify **many unnecessary features**.


In [ ]:

# Prepare the data in COLA format
# Take clients with bad risk (Risk=1) — we want to move them to Risk=0
df_full = pd.concat([X_train, y_train], axis=1)
df_risk_1 = df_full[df_full['Risk'] == 1].sample(8, random_state=42)

print(f"Selected {len(df_risk_1)} clients with bad credit risk (Risk=1)")
print("\nClient profiles:")
display(df_risk_1[numerical_features + categorical_features[:3]].head(4))

In [ ]:
# Initialize the COLA data interface
data = COLAData(
    factual_data=df_risk_1,
    label_column='Risk',
    numerical_features=numerical_features
)

# Initialize the COLA model interface
ml_model = Model(model=pipe, backend="sklearn")

# Generate counterfactuals with DiCE
explainer = DiCE(ml_model=ml_model)

factual, counterfactual = explainer.generate_counterfactuals(
    data=data,
    factual_class=1,                 # the class we are moving FROM
    total_cfs=2,                     # 2 counterfactuals for each instance
    features_to_keep=['Age', 'Sex'], # these features cannot be changed
    continuous_features=numerical_features
)

print(f"Factual examples: {factual.shape[0]}")
print(f"Counterfactual examples: {counterfactual.shape[0]}")

Посмотрим на фактические и контрфактуальные примеры. 

In [ ]:
factual

In [ ]:
counterfactual

Notice that:

1. Age mostly remained untouched, as did sex, saving accounts, and checking account.
2. The examples are arranged somewhat inconveniently (it would be nice, for example, to preserve the indices of the original instances), but intuitively it is clear that they go one after another, that is, for the first instance its counterfactuals are 1 and 2, for the second — 3 and 4, and so on.
3. Credit amount and duration change quite substantially, and more often than the loan purpose.


In [ ]:
# Add the counterfactuals to the data object so we can work with them further.
data.add_counterfactuals(counterfactual, with_target_column=True)
data.summary()

In [ ]:
# Let’s see how many features DiCE changes
def count_changes(factual_df, counterfactual_df, feature_cols):
    """Count the number of changed features for each instance"""
    changes = []
    for i in range(len(factual_df)):
        changed = sum(
            str(factual_df.iloc[i][col]) != str(counterfactual_df.iloc[i][col])
            for col in feature_cols
        )
        changes.append(changed)
    return np.array(changes)

feature_cols = numerical_features + categorical_features
# Take the first counterfactual for each instance
cf_first = counterfactual.iloc[::2, :]
# Take the second counterfactual for each instance
cf_second = counterfactual.iloc[1::2, :]

# count the number of features
changes_first = count_changes(factual, cf_first, feature_cols)
changes_second = count_changes(factual, cf_second, feature_cols)

mean_per_example = (changes_first + changes_second)/2

print(f"   Total number of features: {len(feature_cols)}")
print(f"   Average number of changes per instance: ~{np.mean(mean_per_example)}")

Not that many. Let’s see whether we can reduce it further.


<a id='cola'></a>

## 5. Applying COLA for Sparsification

COLA takes DiCE counterfactuals and **sparsifies** them — keeping only the most important changes.

In [ ]:
# Initialize COLA
from xai_cola.ce_sparsifier import COLA
sparsifier = COLA(
    data=data,
    ml_model=ml_model
)

# Choose the strategy
sparsifier.set_policy(
    matcher="ot",         # Optimal transport for matching
    attributor="pshap",   # p-SHAP for feature attribution
    random_state=42
)

In [ ]:
# Find the minimum number of changes
limited_actions = sparsifier.query_minimum_actions()

print(f'Limited actions count: {limited_actions}')

refined_cf_df = sparsifier.get_refined_counterfactual(limited_actions=limited_actions)
display(refined_cf_df)

In [ ]:
factual

We obtained one counterfactual per instance. Let’s see whether this affected the number of changed features.


In [ ]:
changes_after_cola = count_changes(factual, refined_cf_df, feature_cols)
changes_after_cola, np.mean(changes_after_cola)

Bugs. But this is totally okay :)  

In [ ]:
changes_first, changes_second # but it really did become a bit smaller

The library also includes plots. From this point on, they are just for visualization.

In [ ]:
# Bar chart: number of actions
sparsifier.stacked_bar_chart(save_path='./results')

In [ ]:
# Heatmap of change directions
sparsifier.heatmap_direction(
    save_path=None,
    save_mode='combined',
    show_axis_labels=True
)

Overall, COLA has a beautiful theoretical idea. If you want, you can compare the attributions in COLA with SHAP, or create a naive version of COLA using SHAP. The library itself does not always work perfectly, but it is wonderful that new implementations of different methods exist and keep appearing!

Wishing you successful counterfactuals,

Your data author! :)